In [2]:
import json
import os
from dataclasses import dataclass
from pathlib import Path
from typing import List, Optional, Tuple

import safetensors.torch
from safetensors import safe_open
import torch
from torchtune.modules import RotaryPositionalEmbeddings
from simple_parsing.helpers import Serializable
from torch import nn
from torch.nn import functional as F
import mistral_inf

import error: No module named 'triton'


c:\Users\bapti\Desktop\Code\Pytorch\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
conf = mistral_inf.TransformerArgs(2048, 22, 64, 5632, 32, 4, 1e-5, 32000, 10000, max_batch_size=10)

# Tensors exploration

In [3]:
file_path = "/Users/bapti/Downloads/mistral-7B-v0.3/consolidated.safetensors"

with safe_open(file_path, framework="pt", device=0) as f:
    tensor_list = f.keys()

In [4]:
layers = [[x for x in tensor_list if f'layers.{i}.' in x] for i in range(32)]
other = [x for x in tensor_list if f'layers' not in x]

In [5]:
assert all([len(x)==9 for x in layers])
for tensor in layers[0]:
    print(tensor.removeprefix('layers.0.'))

attention.wk.weight
attention.wo.weight
attention.wq.weight
attention.wv.weight
attention_norm.weight
feed_forward.w1.weight
feed_forward.w2.weight
feed_forward.w3.weight
ffn_norm.weight


In [6]:
for tensor in other:
    print(tensor)

norm.weight
output.weight
tok_embeddings.weight


In [7]:
del other, tensor, layers

# RoPE

In [8]:
x = torch.ones(1, 10, 32, 64)
freqs = mistral_inf.precompute_freqs_cis(64, 10, 10000)
out = mistral_inf.apply_rotary_emb(x, x, freqs)
out[0]

tensor([[[[ 1.0000,  1.0000,  1.0000,  ...,  1.0000,  1.0000,  1.0000],
          [ 1.0000,  1.0000,  1.0000,  ...,  1.0000,  1.0000,  1.0000],
          [ 1.0000,  1.0000,  1.0000,  ...,  1.0000,  1.0000,  1.0000],
          ...,
          [ 1.0000,  1.0000,  1.0000,  ...,  1.0000,  1.0000,  1.0000],
          [ 1.0000,  1.0000,  1.0000,  ...,  1.0000,  1.0000,  1.0000],
          [ 1.0000,  1.0000,  1.0000,  ...,  1.0000,  1.0000,  1.0000]],

         [[-0.3012,  1.3818,  0.0502,  ...,  1.0002,  0.9999,  1.0001],
          [-0.3012,  1.3818,  0.0502,  ...,  1.0002,  0.9999,  1.0001],
          [-0.3012,  1.3818,  0.0502,  ...,  1.0002,  0.9999,  1.0001],
          ...,
          [-0.3012,  1.3818,  0.0502,  ...,  1.0002,  0.9999,  1.0001],
          [-0.3012,  1.3818,  0.0502,  ...,  1.0002,  0.9999,  1.0001],
          [-0.3012,  1.3818,  0.0502,  ...,  1.0002,  0.9999,  1.0001]],

         [[-1.3254,  0.4932, -0.9265,  ...,  1.0004,  0.9997,  1.0003],
          [-1.3254,  0.4932, -

In [3]:
positional_encoder = RotaryPositionalEmbeddings(64, 20)

In [ ]:

(positional_encoder(x)-out[0]).max()

In [10]:
positional_encoder = positional_encoder.to('cuda')

# Norme

In [17]:
mistral_norm = mistral_inf.RMSNorm(2048)
x = torch.rand(1, 10, 2048).to(torch.float16)
mistral_output = mistral_norm(x)

In [26]:
class RMSNorm(torch.nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6, norm=None):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
        if norm is not None:
            self.load(norm)

    def forward(self, x):
        output = x.float()
        norm = torch.rsqrt(output.pow(2).mean(-1, keepdim=True) + self.eps)
        output = (output*norm).type_as(x)
        return output * self.weight
    
    def load(self, norm):
        self.weight = norm.weight


In [27]:
new_norm = RMSNorm(2048, norm=mistral_norm)
new_norm(x)-mistral_output

tensor([[[0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]]], dtype=torch.float16,
       grad_fn=<SubBackward0>)

# Attention

In [11]:
mistral_attn = mistral_inf.Attention(conf).to('cuda')
mistral_attn.wq.weight = torch.nn.Parameter(mistral_attn.wq.weight.to(torch.float16))
mistral_attn.wk.weight = torch.nn.Parameter(mistral_attn.wk.weight.to(torch.float16))
mistral_attn.wv.weight = torch.nn.Parameter(mistral_attn.wv.weight.to(torch.float16))
mistral_attn.wo.weight = torch.nn.Parameter(mistral_attn.wo.weight.to(torch.float16))

In [12]:
x = torch.rand(1, 10, 2048).to('cuda', torch.float16)
positions = torch.tensor([i for i in range(10)]).to('cuda', torch.int64)
freqs = freqs.to('cuda')
mistral_output = mistral_attn(x, freqs, positions, None)

In [13]:
bsz, seqlen, _ = x.shape
xq, xk, xv = mistral_attn.wq(x), mistral_attn.wk(x), mistral_attn.wv(x)
xq = xq.view(bsz, seqlen, mistral_attn.n_heads, mistral_attn.args.head_dim)
xk = xk.view(bsz, seqlen, mistral_attn.n_kv_heads, mistral_attn.args.head_dim)
xv = xv.view(bsz, seqlen, mistral_attn.n_kv_heads, mistral_attn.args.head_dim)
xq, xk = mistral_inf.apply_rotary_emb(xq, xk, freqs_cis=freqs.to('cuda'))

In [14]:
if positions.shape[0] > 1:
    # prefill
    key, value = mistral_inf.repeat_kv(xk, xv, mistral_attn.repeats)
else:
    cur_pos = positions[-1].item() + 1
    key, value = mistral_inf.repeat_kv(
        mistral_attn.cache_k[:bsz, :cur_pos, ...],
        mistral_attn.cache_v[:bsz, :cur_pos, ...],
        mistral_attn.repeats,
    )

query = xq.transpose(1, 2)
key = key.transpose(1, 2)
value = value.transpose(1, 2)
# scores : [bsz, n_heads, seqlen | 1, seqlen]
scores = torch.matmul(query, key.transpose(2, 3)) * mistral_attn.scale
scores = scores.float()
scores = nn.functional.softmax(scores, dim=-1).type_as(query)
output = torch.matmul(scores, value)  # (bs, n_local_heads, slen, head_dim)
output = output.transpose(1, 2).contiguous().view(bsz, seqlen, -1)
mistral_attn.wo(output)-mistral_output

tensor([[[0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]]], device='cuda:0',
       dtype=torch.float16, grad_fn=<SubBackward0>)

In [15]:
attention = F.scaled_dot_product_attention(query, key, value)
mistral_attn.wo(attention.transpose(1, 2).contiguous().view(bsz, seqlen, -1))-mistral_output

tensor([[[ 0.0000e+00,  0.0000e+00, -3.0518e-05,  ...,  0.0000e+00,
           0.0000e+00,  3.0518e-05],
         [ 0.0000e+00,  0.0000e+00, -3.0518e-05,  ...,  0.0000e+00,
          -1.2207e-04,  7.6294e-05],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  1.2207e-04,
          -6.1035e-05, -3.0518e-05],
         ...,
         [-1.2207e-04,  0.0000e+00,  9.1553e-05,  ...,  0.0000e+00,
           0.0000e+00, -1.5259e-05],
         [ 0.0000e+00, -6.1035e-05,  3.0518e-05,  ...,  1.2207e-04,
           0.0000e+00, -1.5259e-05],
         [ 0.0000e+00,  6.1035e-05,  6.1035e-05,  ...,  0.0000e+00,
           0.0000e+00,  7.6294e-05]]], device='cuda:0', dtype=torch.float16,
       grad_fn=<SubBackward0>)

In [4]:
class Attention(nn.Module):
    def __init__(self, attention=None):
        super().__init__()
        self.wq = nn.Linear(in_features=2048, out_features=2048, bias=False)
        self.wk = nn.Linear(in_features=2048, out_features=256, bias=False)
        self.wv = nn.Linear(in_features=2048, out_features=256, bias=False)
        self.wo = nn.Linear(in_features=2048, out_features=2048, bias=False)
        self.Hq, self.H = 32, 4
        if attention is not None:
            self.load(attention)

    def forward(self, x):
        N, S, E = x.shape
        xq = self.wq(x).view(N, S, self.Hq, -1)
        xk = self.wk(x).view(N, S, self.H, -1)
        xv = self.wv(x).view(N, S, self.H, -1)
        xq = positional_encoder(xq)
        xk = positional_encoder(xk)
        attention = F.scaled_dot_product_attention(
            xq.transpose(1, 2),
            xk.transpose(1, 2),
            xv.transpose(1, 2),
            enable_gqa=True,
        )
        return self.wo(attention.transpose(1, 2).contiguous().view(N, S, E))
    
    def load(self, attention):
        self.wq.weight = attention.wq.weight
        self.wk.weight = attention.wk.weight
        self.wv.weight = attention.wv.weight
        self.wo.weight = attention.wo.weight

In [17]:
new_attention = Attention(mistral_attn)
new_attention(x)-mistral_output

tensor([[[ 0.0000e+00,  0.0000e+00, -3.0518e-05,  ...,  0.0000e+00,
           0.0000e+00,  3.0518e-05],
         [ 0.0000e+00,  0.0000e+00, -6.1035e-05,  ...,  0.0000e+00,
           0.0000e+00,  1.5259e-05],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
          -6.1035e-05, -4.5776e-05],
         ...,
         [-1.2207e-04,  0.0000e+00,  9.1553e-05,  ...,  0.0000e+00,
           0.0000e+00,  1.5259e-05],
         [ 0.0000e+00,  0.0000e+00,  3.0518e-05,  ...,  0.0000e+00,
           0.0000e+00,  3.0518e-05],
         [ 0.0000e+00,  6.1035e-05,  3.0518e-05,  ...,  0.0000e+00,
           0.0000e+00,  7.6294e-05]]], device='cuda:0', dtype=torch.float16,
       grad_fn=<SubBackward0>)

# Feed Forward

In [18]:
mistral_ffn = mistral_inf.FeedForward(conf).to('cuda')
mistral_ffn.w1.weight = torch.nn.Parameter(mistral_ffn.w1.weight.to(torch.float16))
mistral_ffn.w2.weight = torch.nn.Parameter(mistral_ffn.w2.weight.to(torch.float16))
mistral_ffn.w3.weight = torch.nn.Parameter(mistral_ffn.w3.weight.to(torch.float16))
mistral_ffn

FeedForward(
  (w1): Linear(in_features=2048, out_features=5632, bias=False)
  (w2): Linear(in_features=5632, out_features=2048, bias=False)
  (w3): Linear(in_features=2048, out_features=5632, bias=False)
)

In [19]:
mistral_output = mistral_ffn(x)

In [5]:
class FeedForward(nn.Module):
    def __init__(self, mlp=None):
        super().__init__()
        self.gate_proj = nn.Linear(in_features=2048, out_features=5632, bias=False)
        self.up_proj = nn.Linear(in_features=2048, out_features=5632, bias=False)
        self.down_proj = nn.Linear(in_features=5632, out_features=2048, bias=False)
        self.act_fn = nn.SiLU()
        if mlp is not None:
            self.load(mlp)

    def forward(self, x):
        x = self.up_proj(x)*self.act_fn(self.gate_proj(x))
        return self.down_proj(x)
    
    def load(self, mlp):
        self.gate_proj.weight = mlp.w1.weight
        self.up_proj.weight = mlp.w3.weight
        self.down_proj.weight = mlp.w2.weight

In [21]:
new_ffn = FeedForward(mistral_ffn)
new_ffn

FeedForward(
  (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
  (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
  (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
  (act_fn): SiLU()
)

In [22]:
new_ffn(x)-mistral_output

tensor([[[0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]]], device='cuda:0',
       dtype=torch.float16, grad_fn=<SubBackward0>)

# Transformer Block

In [23]:
mistral_block = mistral_inf.TransformerBlock(conf).to('cuda')
mistral_block

TransformerBlock(
  (attention): Attention(
    (wq): Linear(in_features=2048, out_features=2048, bias=False)
    (wk): Linear(in_features=2048, out_features=256, bias=False)
    (wv): Linear(in_features=2048, out_features=256, bias=False)
    (wo): Linear(in_features=2048, out_features=2048, bias=False)
  )
  (feed_forward): FeedForward(
    (w1): Linear(in_features=2048, out_features=5632, bias=False)
    (w2): Linear(in_features=5632, out_features=2048, bias=False)
    (w3): Linear(in_features=2048, out_features=5632, bias=False)
  )
  (attention_norm): RMSNorm()
  (ffn_norm): RMSNorm()
)

In [6]:
def convert_block_float16(block):
    attn = block.attention
    attn.wq.weight = torch.nn.Parameter(attn.wq.weight.to(torch.float16))
    attn.wk.weight = torch.nn.Parameter(attn.wk.weight.to(torch.float16))
    attn.wv.weight = torch.nn.Parameter(attn.wv.weight.to(torch.float16))
    attn.wo.weight = torch.nn.Parameter(attn.wo.weight.to(torch.float16))
    feed_forward = block.feed_forward
    feed_forward.w1.weight = torch.nn.Parameter(feed_forward.w1.weight.to(torch.float16))
    feed_forward.w2.weight = torch.nn.Parameter(feed_forward.w2.weight.to(torch.float16))
    feed_forward.w3.weight = torch.nn.Parameter(feed_forward.w3.weight.to(torch.float16))
    return block

In [25]:
mistral_block = convert_block_float16(mistral_block)

In [26]:
mistral_output = mistral_block(x, freqs, positions, None)

In [7]:
class Block(nn.Module):
    def __init__(self, block):
        super().__init__()
        self.attention = Attention(block.attention)
        self.feed_forward = FeedForward(block.feed_forward)
        self.attention_norm = block.attention_norm
        self.ffn_norm = block.ffn_norm

    def forward(self, x):
        x = x + self.attention(self.attention_norm(x))
        x = x + self.feed_forward(self.ffn_norm(x))
        return x

In [28]:
new_block = Block(mistral_block)
new_block

Block(
  (attention): Attention(
    (wq): Linear(in_features=2048, out_features=2048, bias=False)
    (wk): Linear(in_features=2048, out_features=256, bias=False)
    (wv): Linear(in_features=2048, out_features=256, bias=False)
    (wo): Linear(in_features=2048, out_features=2048, bias=False)
  )
  (feed_forward): FeedForward(
    (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
    (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
    (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
    (act_fn): SiLU()
  )
  (attention_norm): RMSNorm()
  (ffn_norm): RMSNorm()
)

In [29]:
new_block(x)-mistral_output

tensor([[[-2.4414e-04,  0.0000e+00,  0.0000e+00,  ..., -1.2207e-04,
           0.0000e+00,  0.0000e+00],
         [-2.4414e-04,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
           0.0000e+00,  0.0000e+00],
         [ 0.0000e+00,  0.0000e+00,  4.8828e-04,  ...,  0.0000e+00,
           0.0000e+00, -2.4414e-04],
         ...,
         [ 4.8828e-04,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
           0.0000e+00, -4.8828e-04],
         [ 2.4414e-04,  2.4414e-04,  9.7656e-04,  ...,  0.0000e+00,
           0.0000e+00, -2.4414e-04],
         [ 6.1035e-05,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
           0.0000e+00,  0.0000e+00]]], device='cuda:0', dtype=torch.float16,
       grad_fn=<SubBackward0>)

# Delete everything

In [41]:
%who

Attention	 Block	 F	 FeedForward	 List	 Optional	 Path	 RotaryPositionalEmbeddings	 Serializable	 
Tuple	 conf	 convert_block_float16	 dataclass	 f	 file_path	 gc	 json	 mistral_inf	 
nn	 os	 positional_encoder	 safe_open	 safetensors	 tensor_list	 torch	 


In [31]:
torch.cuda.memory_allocated()

524068352

In [32]:
del mistral_attn, mistral_block, mistral_ffn, mistral_output, new_attention, new_block, new_ffn, out, output
del attention, bsz, freqs, key, positions, query, scores, seqlen, value, x, xk, xq, xv

In [40]:
torch.cuda.memory_allocated()

356224000

In [35]:
import gc; gc.collect()

328

In [39]:
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

# Model

In [8]:
model = mistral_inf.Transformer(conf).to('cuda')
model

Transformer(
  (tok_embeddings): Embedding(32000, 2048)
  (layers): ModuleList(
    (0-21): 22 x TransformerBlock(
      (attention): Attention(
        (wq): Linear(in_features=2048, out_features=2048, bias=False)
        (wk): Linear(in_features=2048, out_features=256, bias=False)
        (wv): Linear(in_features=2048, out_features=256, bias=False)
        (wo): Linear(in_features=2048, out_features=2048, bias=False)
      )
      (feed_forward): FeedForward(
        (w1): Linear(in_features=2048, out_features=5632, bias=False)
        (w2): Linear(in_features=5632, out_features=2048, bias=False)
        (w3): Linear(in_features=2048, out_features=5632, bias=False)
      )
      (attention_norm): RMSNorm()
      (ffn_norm): RMSNorm()
    )
  )
  (norm): RMSNorm()
  (output): Linear(in_features=2048, out_features=32000, bias=False)
)

In [10]:
torch.cuda.memory_allocated()

14050500608

In [9]:
for block in model.layers:
    block = convert_block_float16(block)
model.tok_embeddings.weight = nn.Parameter(model.tok_embeddings.weight.to(torch.float16))
model.output.weight = nn.Parameter(model.output.weight.to(torch.float16))

In [12]:
torch.cuda.memory_allocated()

11850588160

In [10]:
ids = torch.randint(0, 32000-1, (1, 10)).to('cuda', torch.int64)
positions = torch.tensor([i for i in range(10)]).to('cuda', torch.int64)
with torch.no_grad():
    mistral_output = model(ids, positions)

In [11]:
class Transformers(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.embeddings = model.tok_embeddings
        self.layers = nn.ModuleList(
            [Block(block) for block in model.layers]
        )
        self.norm = model.norm
        self.output = model.output

    def forward(self, input_ids):
        x = self.embeddings(input_ids)
        for layer in self.layers:
            x = layer(x)
        return self.output(self.norm(x))

In [12]:
new_model = Transformers(model)

In [13]:
new_model = new_model.to('cpu')

In [14]:
ids = torch.randint(0, 32000-1, (1, 10)).to('cpu', torch.int64)
positions = torch.tensor([i for i in range(10)]).to('cpu', torch.int64)
with torch.no_grad():
    new_output = new_model(ids)

In [15]:
new_output-mistral_output.to('cpu')

tensor([[[ 0.0591, -1.3208, -0.6169,  ...,  0.8018,  1.4434,  0.7454],
         [ 0.5762, -0.1797, -1.1025,  ...,  0.5327,  0.5160,  0.4128],
         [-0.5114, -0.1907,  0.7383,  ...,  0.5212, -0.7935,  2.0122],
         ...,
         [-1.3015, -0.2860,  1.7622,  ...,  1.5068, -0.5237,  0.5313],
         [ 0.0055,  0.3042,  0.5580,  ...,  1.7246,  0.1581,  1.1353],
         [ 0.0532, -0.4807,  0.4374,  ..., -0.3622,  1.5972,  0.0842]]])